# 05 - Đánh giá và So sánh Models

## Phát hiện Giao dịch Gian lận dựa trên Mạng Neural Đồ thị

Notebook này đánh giá và so sánh hiệu suất của 3 kiến trúc GNN:
- Bảng metrics: Precision, Recall, F1-Score, AUC-ROC, AUC-PR
- Confusion Matrix cho mỗi model
- ROC Curve
- Precision-Recall Curve
- Bảng so sánh tổng hợp

In [ ]:
import sys
sys.path.insert(0, '..')

from src.config import set_seed, DEVICE, HIDDEN_DIM, NUM_LAYERS, NUM_HEADS, DROPOUT
from src.data_loader import IEEECISDataLoader
from src.graph_builder import HeteroGraphBuilder
from src.sampling import ImbalanceSampler
from src.models import HeteroGraphSAGE, HeteroGAT, HeteroRGCN
from src.trainer import GNNTrainer
from src.evaluator import ModelEvaluator

set_seed(42)
print(f"Device: {DEVICE}")

## 1. Chuẩn bị dữ liệu và train models

> **Lưu ý**: Nếu đã chạy notebook 04, có thể load best models từ `models/` thay vì train lại.

In [ ]:
# Load data
loader = IEEECISDataLoader()
df = loader.load()

# Build graph
builder = HeteroGraphBuilder(df)
data = builder.build()

# Create loaders
sampler = ImbalanceSampler(data)
loaders = sampler.get_all_loaders()
loss_fn = sampler.get_focal_loss()

metadata = data.metadata()
in_channels = data['txn'].x.shape[1]

## 2. Train (hoặc load) các models

In [ ]:
import torch
from pathlib import Path

models_config = {
    'HeteroSAGE': {
        'class': HeteroGraphSAGE,
        'kwargs': dict(metadata=metadata, in_channels=in_channels,
                       hidden_channels=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT),
    },
    'HeteroGAT': {
        'class': HeteroGAT,
        'kwargs': dict(metadata=metadata, in_channels=in_channels,
                       hidden_channels=HIDDEN_DIM, num_layers=NUM_LAYERS,
                       num_heads=NUM_HEADS, dropout=DROPOUT),
    },
    'HeteroRGCN': {
        'class': HeteroRGCN,
        'kwargs': dict(metadata=metadata, in_channels=in_channels,
                       hidden_channels=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT),
    },
}

trainers = {}
for name, cfg in models_config.items():
    set_seed(42)
    model = cfg['class'](**cfg['kwargs'])
    
    # Try to load saved model
    saved_path = Path(f'../models/{name}_best.pt')
    if saved_path.exists():
        print(f"Loading saved model: {saved_path}")
        checkpoint = torch.load(str(saved_path), map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        model = model.to(DEVICE)
        trainer = GNNTrainer(model, loss_fn, model_name=name)
    else:
        print(f"Training {name} from scratch...")
        trainer = GNNTrainer(model, loss_fn, model_name=name)
        trainer.train(loaders['train'], loaders['val'])
    
    trainers[name] = trainer
    print()

## 3. Đánh giá trên Test Set

In [ ]:
evaluator = ModelEvaluator()

results = {}
for name, trainer in trainers.items():
    y_true, y_pred, y_prob = trainer.predict(loaders['test'])
    metrics = evaluator.evaluate(y_true, y_pred, y_prob, model_name=name)
    results[name] = {
        'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob, 'metrics': metrics
    }

## 4. Bảng so sánh tổng hợp

In [ ]:
evaluator.plot_comparison()

## 5. Hiển thị tất cả biểu đồ đánh giá

In [ ]:
from IPython.display import Image, display
from pathlib import Path

metrics_dir = Path('../output/metrics')
for img_path in sorted(metrics_dir.glob('*.png')):
    print(f"\n--- {img_path.stem} ---")
    display(Image(filename=str(img_path), width=700))

## 6. Kết luận

Bảng tổng kết hiệu suất của 3 kiến trúc GNN trên bài toán phát hiện gian lận IEEE-CIS:

| Metric | HeteroSAGE | HeteroGAT | HeteroRGCN |
|--------|-----------|-----------|------------|
| Precision | - | - | - |
| Recall | - | - | - |
| F1-Score | - | - | - |
| AUC-ROC | - | - | - |
| AUC-PR | - | - | - |

*Điền kết quả sau khi chạy training.*